# SaafBol pipeline test on JFLEG (validation + test)
Runs the **same pipeline as `cleanspeak_cli.py`** (typo model included) on JFLEG, which has 4 human corrections per sentence.
Use **Runtime > Change runtime type > T4 GPU** (CPU also works, just slowly). Run the cells top to bottom.

In [ ]:
# CELL 1: install (Colab already has torch and transformers)
!pip install -q peft sentencepiece better-profanity sacrebleu datasets accelerate

In [ ]:
# CELL 2: log in to Hugging Face (the LoRA adapter repo is private; a "read" token is enough)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# CELL 3: imports and settings
import os, re, csv, time, random, difflib
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

N = 300             # how many JFLEG sentences to test (validation + test = 1,503 in total). Use None for all of them
BATCH = 16          # sentences per batch (use 4 on CPU)
USE_SPELL = True    # the A/B cells below flip this

BASE = "grammarly/coedit-large"
ADAPTER = "Shinjan123/coedit-paradetox-lora"
SPELL_MODEL = "oliverguhr/spelling-correction-english-base"
PREFIX = "Clean this sentence: "   # same instruction the adapter was trained with

In [ ]:
# CELL 4: word lists and rule-based helpers (copied unchanged from cleanspeak_cli.py)
# ---------------------------------------------------------------- word lists (edit these freely)
# Slang: deterministic and explainable. Longest keys are matched first, so "lit af" wins over "af".
SLANG = {"lit af": "really exciting", "tbh": "honestly", "idk": "I do not know", "gonna": "going to",
         "wanna": "want to", "gotta": "have to", "kinda": "kind of", "u": "you", "ur": "your",
         "pls": "please", "thx": "thanks", "cuz": "because", "af": "very", "bro": "", "dude": "",
         "lol": "", "irl": "in real life", "bruh": "", "bruv": ""}

# Words that sit on the profanity list but are not toxic by themselves (a judgment call: say so in the report).
IGNORE = {"pot", "gay", "omg", "kill", "dang", "wtf"}

# Mild insults the default list misses, so detection catches them.
EXTRA_BAD = ["rascal", "bastard", "idiot", "moron", "clown", "stupid", "jerk"]

# Backstop: used ONLY when the model leaves a bad word behind. Each replacement is milder but keeps the
# criticism (an insult stays a mild criticism, never praise). Crude on purpose: it is a safety net.
SOFTEN = {"idiot": "fool", "idiots": "fools", "moron": "fool", "morons": "fools", "stupid": "unwise",
          "dumb": "unwise", "jerk": "unpleasant person", "bastard": "unpleasant person",
          "rascal": "troublemaker", "clown": "fool", "sucked": "performed poorly", "sucks": "is bad",
          "sucking": "performing poorly", "screwed": "messed", "crap": "nonsense", "damn": "darn",
          "hell": "heck", "shit": "nonsense", "bullshit": "nonsense"}

from better_profanity import profanity  # noqa: E402

profanity.load_censor_words()            # built-in list first...
profanity.add_censor_words(EXTRA_BAD)    # ...then ADD ours (load_censor_words(x) would REPLACE the defaults)

SLANG_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r")\b", re.I)


# ---------------------------------------------------------------- rule-based helpers
def fix_slang(text):
    s = SLANG_RE.sub(lambda m: SLANG[m.group(0).lower()], text)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.!?])", r"\1", s)            # no space before punctuation
    return re.sub(r"^[\s,]+|[\s,]+$", "", s)        # trim commas left where bro/dude was removed


def real_dirty(text):
    """Bad words that remain, ignoring the IGNORE list. Checked word by word so 'assess' is NOT flagged."""
    return [w for w in re.findall(r"[\w']+", text)
            if w.lower() not in IGNORE and profanity.contains_profanity(w)]


def soften_words(text):
    def sub(m):
        w = m.group(0)
        rep = SOFTEN.get(w.lower())
        if not rep:
            return w
        return rep.capitalize() if w[:1].isupper() else rep   # keep the capital letter
    return re.sub(r"[A-Za-z']+", sub, text)


def mask_dirty(text):
    """Last resort: keep the first letter, star the rest (r******)."""
    dirty = {w.lower() for w in real_dirty(text)}
    return re.sub(r"[\w']+", lambda m: m.group(0)[0] + "*" * (len(m.group(0)) - 1)
                  if m.group(0).lower() in dirty else m.group(0), text)


def split_sentences(text):
    return [p for p in re.split(r"(?<=[.!?])\s+", text.strip()) if p]

In [ ]:
# CELL 5: load the three models (about 4 GB download the first time)
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

TOK = AutoTokenizer.from_pretrained(BASE)
MODEL = PeftModel.from_pretrained(AutoModelForSeq2SeqLM.from_pretrained(BASE), ADAPTER).to(DEVICE).eval()
SP_TOK = AutoTokenizer.from_pretrained(SPELL_MODEL)
SP_MODEL = AutoModelForSeq2SeqLM.from_pretrained(SPELL_MODEL).to(DEVICE).eval()
print("models loaded")

In [ ]:
# CELL 6: model functions and the one-sentence pipeline (copied unchanged from cleanspeak_cli.py)
def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).split()

def generate(texts, prefix=PREFIX, beams=4):
    import torch
    enc = TOK([prefix + t for t in texts], return_tensors="pt", padding=True,
              truncation=True, max_length=96).to(DEVICE)
    with torch.no_grad():
        out = MODEL.generate(**enc, max_length=96, num_beams=beams)
    return TOK.batch_decode(out, skip_special_tokens=True)


def polish(text):
    with MODEL.disable_adapter():                   # adapter off = original CoEdIT, so grammar skills are untouched
        return generate([text], prefix="Fix the grammar: ")[0]


def spell_fix(text):
    """The typo model: fixes misspellings, repeated letters, and split / merged words."""
    import torch
    enc = SP_TOK(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        out = SP_MODEL.generate(**enc, max_length=128, num_beams=4)
    return SP_TOK.decode(out[0], skip_special_tokens=True)


def normalize_caps(text):
    """ALL-CAPS input (DANG IT) is unusual for T5, so the model leaves it alone. Lowercase it first."""
    letters = [c for c in text if c.isalpha()]
    # mostly-uppercase and at least 2 words, so "NASA launched a rocket", "OK", and "PDF" are left alone
    if len(letters) >= 4 and len(text.split()) >= 2 and sum(c.isupper() for c in letters) / len(letters) > 0.8:
        return text.lower()
    return text


# ---------------------------------------------------------------- the pipeline (v1 + typo model on top)
def cleanspeak_sentence(text):
    """One sentence in -> (final sentence, dict of every stage for debugging)."""
    stages = {"input": text}
    s = fix_slang(normalize_caps(text))
    stages["1 slang"] = s
    if not s:
        return "", stages

    if USE_SPELL:                                   # typo model, right after the slang dictionary so slang is not "corrected"
        s = spell_fix(s)
        stages["1b spell"] = s

    out = generate([s])[0]                          # the model sees every sentence
    stages["2 model"] = out

    if real_dirty(out):                             # the model missed a bad word -> deterministic backstop
        out = soften_words(out)
        stages["3 backstop"] = out

    out = polish(out)
    stages["4 polish"] = out

    if real_dirty(out):                             # still dirty -> mask rather than let it through
        out = mask_dirty(out)
        stages["5 mask"] = out
    return out, stages


def cleanspeak(text):
    """Any text in (one or many sentences) -> (final text, list of stage dicts)."""
    outs, all_stages = [], []
    for sent in split_sentences(text):              # the model was trained on single sentences
        o, st = cleanspeak_sentence(sent)
        outs.append(o)
        all_stages.append(st)
    return " ".join(o for o in outs if o), all_stages


def changes(a, b):
    d = [x for x in difflib.ndiff(a.split(), b.split()) if x[0] in "+-"]
    return "  ".join(d) if d else "(no change)"

In [ ]:
# CELL 7: the SAME pipeline, but every model stage runs on batches (much faster on a GPU)
def spell_fix_batch(texts):
    enc = SP_TOK(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        out = SP_MODEL.generate(**enc, max_length=128, num_beams=4)
    return SP_TOK.batch_decode(out, skip_special_tokens=True)


def polish_batch(texts):
    with MODEL.disable_adapter():                   # adapter off = original CoEdIT
        return generate(texts, prefix="Fix the grammar: ")


def _batched(fn, items, bs):
    """Apply fn (list -> list, same length) in chunks, longest first so padding is minimal."""
    order = sorted(range(len(items)), key=lambda i: len(items[i]), reverse=True)
    out = [None] * len(items)
    for k in range(0, len(order), bs):
        chunk = order[k:k + bs]
        for i, r in zip(chunk, fn([items[i] for i in chunk])):
            out[i] = r
    return out


def cleanspeak_batch(texts, bs=16):
    """Same logic as cleanspeak() / cleanspeak_sentence(), stage by stage on batches."""
    parts, owner = [], []
    for i, t in enumerate(texts):                   # split into sentences, remember which text each came from
        for sent in split_sentences(t):
            parts.append(sent)
            owner.append(i)
    s = [fix_slang(normalize_caps(p)) for p in parts]
    idx = [i for i, x in enumerate(s) if x]         # sentences that are not empty after the slang stage
    cur = [s[i] for i in idx]
    if USE_SPELL:
        cur = _batched(spell_fix_batch, cur, bs)
    cur = _batched(lambda b: generate(b), cur, bs)
    cur = [soften_words(x) if real_dirty(x) else x for x in cur]    # backstop
    cur = _batched(polish_batch, cur, bs)
    cur = [mask_dirty(x) if real_dirty(x) else x for x in cur]      # last resort
    done = [""] * len(parts)
    for i, x in zip(idx, cur):
        done[i] = x
    res = [[] for _ in texts]
    for o, x in zip(owner, done):
        if x:
            res[o].append(x)
    return [" ".join(r) for r in res]


# consistency check: one-at-a-time and batched must give the same text
CHECK = ["bro i is gonna go there, its lit af", "I recieved ur mesage yesterday and it was realy helpfull",
         "this is bullshit, the teacher dont no nothing. you are an idiot!", "The meeting starts at 10 AM tomorrow.", "DANG IT"]
single = [cleanspeak(t)[0] for t in CHECK]
batch = cleanspeak_batch(CHECK, bs=4)
for t, a, b in zip(CHECK, single, batch):
    print("OK  " if a == b else "DIFF", "|", t, "\n      single:", a, "\n      batch :", b)

In [ ]:
# CELL 8: load JFLEG (validation + test) and put the text back to normal (JFLEG is tokenised: "energy ." / "It 's")
from datasets import load_dataset, concatenate_datasets

raw = load_dataset("jhu-clsp/jfleg")
jfleg = concatenate_datasets([raw["validation"], raw["test"]])          # 755 + 748 sentences
print(len(jfleg), "sentences, columns:", jfleg.column_names)            # sentence, corrections (4 human versions)

def detok(s):
    s = re.sub(r"\s+([.,!?;:%)\]])", r"\1", s)
    s = re.sub(r"([(\[])\s+", r"\1", s)
    s = re.sub(r"\s+('s|'re|'m|'ll|'ve|'d|n't)\b", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

rows = [{"input": detok(r["sentence"]), "refs": [detok(c) for c in r["corrections"]]} for r in jfleg]
rows = [r for r in rows if 3 <= len(r["input"].split()) <= 60]
random.Random(0).shuffle(rows)
if N:
    rows = rows[:N]
print(len(rows), "sentences selected")

In [ ]:
# CELL 9: scoring. chrF = character n-gram F-score against the 4 human corrections (0-100, higher is better)
from sacrebleu.metrics import CHRF
chrf = CHRF()

def evaluate(rows, use_spell):
    global USE_SPELL
    USE_SPELL = use_spell
    t0 = time.time()
    outs = cleanspeak_batch([r["input"] for r in rows], bs=BATCH)
    secs = time.time() - t0
    res = []
    for r, out in zip(rows, outs):
        src, refs = r["input"], r["refs"]
        c_in, c_out = chrf.sentence_score(src, refs).score, chrf.sentence_score(out, refs).score
        res.append({"input": src, "output": out, "ref": refs[0],
                    "chrf_in": round(c_in, 1), "chrf_out": round(c_out, 1), "delta": round(c_out - c_in, 1),
                    "exact_in": int(any(norm(src) == norm(x) for x in refs)),      # input was already acceptable
                    "exact_out": int(any(norm(out) == norm(x) for x in refs)),
                    "bad_in": int(bool(real_dirty(src))), "bad_out": int(bool(real_dirty(out)))})
    print(f"typo model {'ON ' if use_spell else 'OFF'}: {len(rows)} sentences in {secs:.0f}s ({secs / len(rows):.2f}s each)")
    return res

def summarize(res):
    n = len(res)
    better, worse = sum(r["delta"] > 1 for r in res), sum(r["delta"] < -1 for r in res)
    already = [r for r in res if r["exact_in"]]
    return {"avg chrF, raw input": sum(r["chrf_in"] for r in res) / n,
            "avg chrF, our output": sum(r["chrf_out"] for r in res) / n,
            "closer to human refs": better, "further from refs": worse, "about the same": n - better - worse,
            "output equals a human ref": sum(r["exact_out"] for r in res),
            "already-correct sentences": len(already),
            "  ...changed by us (over-editing)": sum(norm(r["output"]) != norm(r["input"]) for r in already),
            "bad words in input": sum(r["bad_in"] for r in res), "bad words left": sum(r["bad_out"] for r in res)}

In [ ]:
# CELL 10: run A, typo model ON
res_on = evaluate(rows, use_spell=True)

In [ ]:
# CELL 11: run B, typo model OFF (same sentences, same everything else)
res_off = evaluate(rows, use_spell=False)

In [ ]:
# CELL 12: side-by-side comparison
a, b = summarize(res_on), summarize(res_off)
print(f"{'':40}{'typo ON':>10}{'typo OFF':>10}")
for k in a:
    fmt = (lambda v: f"{v:10.1f}") if isinstance(a[k], float) else (lambda v: f"{v:10d}")
    print(f"{k:40}{fmt(a[k])}{fmt(b[k])}")

print("\nSentences where the typo model changed the result the most:")
pairs = sorted(zip(res_on, res_off), key=lambda p: abs(p[0]["delta"] - p[1]["delta"]), reverse=True)
shown = 0
for on, off in pairs:
    if on["output"] == off["output"]:
        continue
    print(f"IN : {on['input']}\nON : {on['output']}   (delta {on['delta']:+.1f})\nOFF: {off['output']}   (delta {off['delta']:+.1f})\nREF: {on['ref']}\n")
    shown += 1
    if shown == 10:
        break

In [ ]:
# CELL 13: the worst regressions with the typo model ON (read these by eye, they matter more than the averages)
for r in sorted(res_on, key=lambda x: x["delta"])[:10]:
    print(f"delta {r['delta']:+.1f}\n  IN : {r['input']}\n  OUT: {r['output']}\n  REF: {r['ref']}\n")

In [ ]:
# CELL 14: save everything (and download it if you are on Colab)
def save(res, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(res[0]))
        w.writeheader(); w.writerows(res)

save(res_on, "jfleg_spell_on.csv"); save(res_off, "jfleg_spell_off.csv")
try:
    from google.colab import files
    files.download("jfleg_spell_on.csv"); files.download("jfleg_spell_off.csv")
except Exception:
    print("saved jfleg_spell_on.csv and jfleg_spell_off.csv in the current folder")